In [ ]:
def generate_contact_map(fasta_path, sequence):
    """
    Executes an external ESM-2 script to generate a contact map from a FASTA file.
    The resulting CSV is stored in results_esm2/contact_maps/. If the file already
    exists, the computation is skipped.
    """

    contact_maps_dir = os.path.join("predicted_contact_maps")
    os.makedirs(contact_maps_dir, exist_ok=True)
    
    csv_path = os.path.join(contact_maps_dir, f"{sequence}.csv")
    
    if os.path.exists(csv_path):
        print(f"Skipping: contact map already exists at {csv_path}")
        return csv_path
    
    python_env = "/home/biocomp/anaconda3/envs/esmfold/bin/python"
    script_path = os.path.join("esm2", "get_contact_map.py")
    
    print(f"Running ESM-2 contact map generation for sequence: {sequence}")
    
    try:
        subprocess.run([python_env, script_path, fasta_path, csv_path], check=True)
        print(f"Contact map successfully generated at: {csv_path}")
        return csv_path
        
    except subprocess.CalledProcessError:
        print("Error: ESM-2 script execution failed.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    
    return None

In [ ]:
import os
import glob
import subprocess
from pathlib import Path

def process_all_fastas(input_folder):
    fasta_files = glob.glob(os.path.join(input_folder, "*.fasta"))
    
    if not fasta_files:
        print(f"No se encontraron archivos .fasta en {input_folder}")
        return

    print(f"Se encontraron {len(fasta_files)} archivos. Iniciando procesamiento...")

    for fasta_path in fasta_files:

        sequence_name = Path(fasta_path).stem
        
        generate_contact_map(fasta_path, sequence_name)


In [ ]:
directorio_fastas = "fasta_files"
process_all_fastas(directorio_fastas)

In [ ]:
import os
import pandas as pd
import glob

def summarize_contact_maps(directory="predicted_contact_maps"):
    # Lista para almacenar los datos de cada archivo
    summary_data = []
    
    # Buscamos todos los archivos CSV generados
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    
    if not csv_files:
        print(f"No se encontraron archivos CSV en {directory}")
        return
    
    print(f"Procesando {len(csv_files)} archivos...")

    for file_path in csv_files:
        try:
            # Leer el CSV (asumiendo que no tiene encabezados si es una matriz pura)
            # Si tu script de ESM2 guarda con headers, usa: pd.read_csv(file_path)
            df = pd.read_csv(file_path)
            
            # 1. Nombre de la secuencia (nombre del archivo sin extensión)
            sequence_name = os.path.basename(file_path).replace(".csv", "")
            
            # 2. Longitud (número de filas)
            length = len(df)
            
            # 3. Probabilidades mayores a cero
            # Convertimos a numérico por si acaso y contamos valores > 0
            count_positive = (df > 0).sum().sum()
            
            summary_data.append({
                "Nombre de la secuencia": sequence_name,
                "Longitud": length,
                "# Probabilidades > 0": count_positive
            })
            
        except Exception as e:
            print(f"Error procesando {file_path}: {e}")

    # Crear el DataFrame final
    summary_df = pd.DataFrame(summary_data)
    
    # Opcional: Ordenar por nombre
    summary_df = summary_df.sort_values("Nombre de la secuencia")
    
    # Guardar a un archivo CSV final
    output_name = "resumen_contactos.csv"
    summary_df.to_csv(output_name, index=False)
    
    print(f"\nTabla generada exitosamente: {output_name}")
    print(summary_df.head()) # Mostrar las primeras filas en consola

summarize_contact_maps()

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def summarize_contact_maps_advanced(directory="predicted_contact_maps"):
    summary_data = []
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    
    if not csv_files:
        print(f"No se encontraron archivos CSV en {directory}")
        return
    
    print(f"Procesando {len(csv_files)} archivos...")

    for file_path in csv_files:
        try:
            # 1. Leer el CSV y convertirlo a matriz de NumPy para cálculos rápidos
            df = pd.read_csv(file_path)
            mat = df.to_numpy()
            
            sequence_name = os.path.basename(file_path).replace(".csv", "")
            length = len(df)
            
            # 2. Crear una matriz de distancias |i - j|
            # Esto genera una matriz NxN donde cada celda indica la distancia en la secuencia
            indices = np.arange(length)
            dist_matrix = np.abs(indices[:, None] - indices)
            print(dist_matrix)
            # 3. Crear las máscaras lógicas para las probabilidades > 0
            is_positive = (mat > 0)
            
            # Totales
            count_total = is_positive.sum()
            
            # Sin diagonal: |i - j| > 0
            count_no_diag = (is_positive & (dist_matrix > 0)).sum()
            
            # Sin distancia 1 (se excluye c_i y c_i+1): |i - j| > 1
            count_dist_gt_1 = (is_positive & (dist_matrix > 1)).sum()
            
            # Sin distancia 2 (se excluye hasta c_i+2): |i - j| > 2
            count_dist_gt_2 = (is_positive & (dist_matrix > 2)).sum()
            
            # Sin distancia 3 (se excluye hasta c_i+3): |i - j| > 3
            count_dist_gt_3 = (is_positive & (dist_matrix > 3)).sum()

            # Sin distancia 4 (se excluye hasta c_i+4): |i - j| > 4
            count_dist_gt_4 = (is_positive & (dist_matrix > 4)).sum()
            
            # Agregar a los resultados
            summary_data.append({
                "Nombre de la secuencia": sequence_name,
                "Longitud": length,
                "# Prob > 0 (Total)": count_total,
                "# Prob > 0 (Sin diagonal)": count_no_diag,
                "# Prob > 0 (Distancia > 1)": count_dist_gt_1,
                "# Prob > 0 (Distancia > 2)": count_dist_gt_2,
                "# Prob > 0 (Distancia > 3)": count_dist_gt_3,
                "# Prob > 0 (Distancia > 4)": count_dist_gt_4
            })
            
        except Exception as e:
            print(f"Error procesando {file_path}: {e}")

    # Crear el DataFrame y exportar
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values("Nombre de la secuencia")
    
    output_name = "resumen_contactos_detallado.csv"
    summary_df.to_csv(output_name, index=False)
    
    print(f"\nTabla detallada generada exitosamente: {output_name}")
    print(summary_df.head())

summarize_contact_maps_advanced()

In [ ]:
def plot_fitness_evolution(target_protein, alg_folder="alg_a"):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    # 1. Define paths
    esm2_dir = "results_esm2"
    target_dir = os.path.join(esm2_dir, alg_folder)
    stats_file = f"statistics_{target_protein}.csv"
    stats_path = os.path.join(target_dir, stats_file)

    # 2. Check file exists
    if not os.path.exists(stats_path):
        print(f"Error: File not found at {stats_path}")
        return

    # 3. Load data
    df = pd.read_csv(stats_path)

    # 4. Create plot
    plt.figure(figsize=(10, 6))

    plt.plot(df['generation'], df['best_fitness'],
             marker='8', linewidth=2, markersize=3,
             color='#237227',
             label='Best Fitness')

    plt.plot(df['generation'], df['median_fitness'],
             marker='8', linewidth=2, markersize=4,
             color='#261CC1',
             label='Median Fitness')

    # 5. Format plot
    plt.title(
        f'Best vs Median fitness - {target_protein.upper()}',
        fontsize=14,
        fontweight='bold'
    )
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Fitness', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)

    # 6. Save plot
    plot_file = f"plot_evolution_best_median_{target_protein}.png"
    plot_path = os.path.join(target_dir, plot_file)

    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved at:\n{plot_path}")

    plt.close()

In [ ]:
plot_fitness_evolution("1p9n", alg_folder="alg_a")
#median_fitness

In [3]:
import numpy as np

# Cargar archivos usando numpy (asumiendo que son solo números)
# delimiter=',' indica que es un CSV
# skiprows=1 se usa si tus archivos tienen una fila de encabezado (X, Y, Z)
m1 = np.loadtxt('QGKYKIGNGVTITARQVNGRQEITLEGA.csv', delimiter=',', skiprows=1)
m2 = np.loadtxt('1s7m.csv', delimiter=',', skiprows=1)

# Cálculo del RMSD directo
diff = m1 - m2
rmsd = np.sqrt(np.mean(np.sum(diff**2, axis=1)))

print(f"RMSD: {rmsd}")

RMSD: 13.900738276452532


In [ ]:
def get_sequence_rmsd(sequence, target_protein, main_dir="."):

    alphafold_dir = os.path.join(main_dir, "results_alphaFold")
    os.makedirs(alphafold_dir, exist_ok=True)
    
    dist_matrix_file = os.path.join(alphafold_dir, f"{sequence}.csv")
    real_dist_path = os.path.join(main_dir, "real_distances", f"{target_protein}.csv")
    
    # 1. Validar matriz real
    if not os.path.exists(real_dist_path):
        raise FileNotFoundError(f"No existe la matriz real: {real_dist_path}")
    mat_target = pd.read_csv(real_dist_path).to_numpy()
    
    # 2. Ejecutar AlphaFold si no hay caché
    if not os.path.exists(dist_matrix_file):
        fasta_path = os.path.join(main_dir, f"{sequence}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{sequence}\n{sequence}")
            
        custom_env = os.environ.copy()
        custom_env["MPLBACKEND"] = "Agg"
        custom_env["TF_CPP_MIN_LOG_LEVEL"] = "3"
        custom_env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
        
        try:
            # Al usar check=True, si AlphaFold falla (ej. Out of Memory), 
            # la función lanzará un CalledProcessError inmediatamente.
            subprocess.run([
                colabfold_path, fasta_path, alphafold_dir,
                "--num-recycle", "3", "--use-gpu-relax"
            ], check=True, env=custom_env)

            # Verificar que el PDB realmente se creó
            pdb_files = glob.glob(os.path.join(alphafold_dir, f"{sequence}*rank_001*.pdb"))
            if not pdb_files:
                raise FileNotFoundError(f"AlphaFold no generó el PDB para {sequence}")

            # Procesar el PDB generado
            coords_pred = get_representative_coords_from_pdb(pdb_files[0])
            N = len(coords_pred)
            matrix_pred = np.zeros((N, N))
            for i in range(N):
                for j in range(N):
                    matrix_pred[i, j] = np.linalg.norm(coords_pred[i] - coords_pred[j])
            
            # Guardar matriz final
            column_names = [f"c{i+1}" for i in range(N)]
            df_matrix = pd.DataFrame(matrix_pred, columns=column_names)
            df_matrix.to_csv(dist_matrix_file, index=False)

        finally:
            # Limpieza total de archivos temporales (se ejecuta incluso si hay error)
            for item in glob.glob(os.path.join(alphafold_dir, f"{sequence}*")):
                if not item.endswith(f"{sequence}.csv"):
                    if os.path.isdir(item): shutil.rmtree(item)
                    else: os.remove(item)
            
            for extra in ["cite.bibtex", "config.json", "log.txt"]:
                f_extra = os.path.join(alphafold_dir, extra)
                if os.path.exists(f_extra): os.remove(f_extra)
            if os.path.exists(fasta_path): os.remove(fasta_path)

    # 3. Cálculo final del RMSD (Lanza error si las dimensiones no coinciden)
    mat_pred = pd.read_csv(dist_matrix_file).to_numpy()
    
    if mat_pred.shape != mat_target.shape:
        raise ValueError(f"Dimensiones incompatibles: Predicha {mat_pred.shape} vs Real {mat_target.shape}")
    
    rmsd = np.sqrt(np.mean((mat_pred - mat_target) ** 2))
    return float(rmsd)

get_contact_map antiguo

In [ ]:
import torch
import esm
import numpy as np
import pandas as pd  
import sys
import os

def predict_esm2_contacts(fasta_file, output_csv):
    """
    Predicts a residue–residue contact matrix from a protein sequence
    using the ESM-2 model and saves it as a CSV file.
    """
    
    # 1. Read the FASTA file
    with open(fasta_file, 'r') as f:
        lines = f.readlines()
        
    sequence = ""
    for line in lines:
        if not line.startswith(">"):
            sequence += line.strip()
            
    seq_length = len(sequence)
    
    # 2. Load the ESM-2 model (33-layer version)
    print("Loading ESM-2 model (this may take a few seconds)...")
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    
    # 3. Move model to GPU if available
    if torch.cuda.is_available():
        model = model.cuda()
        
    model.eval()
    
    # 4. Prepare input data
    data = [("protein", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    
    if torch.cuda.is_available():
        batch_tokens = batch_tokens.cuda()
        
    # 5. Predict contacts using attention-based outputs
    print(f"Predicting contacts for sequence of length {seq_length}...")
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=True)
        
    # 6. Extract contact matrix
    contacts = results["contacts"][0].cpu().numpy()
    matrix_size = contacts.shape[0]
    
    # 7. Save matrix with strict formatting (headers, no empty rows)
    column_names = [f"c{i+1}" for i in range(matrix_size)]
    df_matrix = pd.DataFrame(contacts, columns=column_names)
    
    csv_text = df_matrix.to_csv(index=False, header=True, float_format="%.4f")
    
    # === Strict cleanup to remove any empty lines ===
    csv_lines = csv_text.split('\n')
    valid_lines = [line for line in csv_lines if line.strip() != ""]
    final_text = '\n'.join(valid_lines)
    
    with open(output_csv, 'w') as f:
        f.write(final_text)
        
    print(f"Success! ESM-2 matrix ({matrix_size}x{matrix_size}) saved with headers to: {output_csv}")


if __name__ == "__main__":
    if len(sys.argv) < 3:
        print("Usage: python generate_esm2_matrix.py <input.fasta> <output.csv>")
        sys.exit(1)
        
    fasta_in = sys.argv[1]
    csv_out = sys.argv[2]
    predict_esm2_contacts(fasta_in, csv_out)

